### Step1: Load POWER dataset

In [2]:
import pandas as pd
import numpy as np

# Path to power txt file
file_path = './data/household_power_consumption.txt'
# Load with ; separator, handle '?' as NaN
data = pd.read_csv(file_path, sep=';', na_values='?')

# Combine Date and Time to datetime
data['datetime'] = pd.to_datetime(data['Date'] + ' ' + data['Time'], format='%d/%m/%Y %H:%M:%S')
# Numerical timestamp: seconds since the earliest date
min_dt = data['datetime'].min()
data['timestamp'] = (data['datetime'] - min_dt).dt.total_seconds()

# Relevant columns (drop Date/Time/datetime, keep numerics)
cols = ['timestamp', 'Global_active_power', 'Global_reactive_power', 'Voltage', 
        'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
data = data[cols].dropna()  # ~1.25% missing, drop for simplicity
full_data_size = len(data)
# Define dimensions (7D) and their min/max
dimensions = ['timestamp', 'Global_reactive_power', 'Voltage', 'Global_intensity', 
              'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
agg_col = 'Global_active_power'

print(f"Dataset loaded: {full_data_size} rows")
print(f"First 5 data: {data.head()}")

Dataset loaded: 2049280 rows
First 5 data:    timestamp  Global_active_power  Global_reactive_power  Voltage  \
0        0.0                4.216                  0.418   234.84   
1       60.0                5.360                  0.436   233.63   
2      120.0                5.374                  0.498   233.29   
3      180.0                5.388                  0.502   233.74   
4      240.0                3.666                  0.528   235.68   

   Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  
0              18.4             0.0             1.0            17.0  
1              23.0             0.0             1.0            16.0  
2              23.0             0.0             2.0            17.0  
3              23.0             0.0             1.0            17.0  
4              15.8             0.0             1.0            17.0  


### Step2: Create a Small Offline Sample

In [3]:
sample_size = 2000
sample = data.sample(n=sample_size, random_state=42).copy()
print(f"Sample created: {sample.shape[0]} rows")
print(sample.head())

Sample created: 2000 rows
          timestamp  Global_active_power  Global_reactive_power  Voltage  \
1030580  61834800.0                1.502                  0.074   240.17   
1815       108900.0                0.374                  0.264   245.50   
1295977  77758620.0                0.620                  0.300   239.85   
206669   12400140.0                0.280                  0.200   235.72   
1048893  62933580.0                1.372                  0.054   243.95   

         Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  
1030580               6.4             0.0             0.0            18.0  
1815                  1.8             0.0             2.0             0.0  
1295977               3.0             0.0             1.0             1.0  
206669                1.4             0.0             0.0             0.0  
1048893               5.6             0.0             0.0            18.0  


### Step 3: Generate Query Log
For AQP++ to select q_old
(The Query Log should be same to LAQP)


In [4]:
from query_generate import generate_random_query

# generate a random query to test and see min, max value.
q = generate_random_query(data, dimensions, test = True)
print(q)

dim: timestamp, min value: 0.0, max value: 124515480.0
dim: Global_reactive_power, min value: 0.0, max value: 1.39
dim: Voltage, min value: 223.2, max value: 254.15
dim: Global_intensity, min value: 0.2, max value: 48.4
dim: Sub_metering_1, min value: 0.0, max value: 88.0
dim: Sub_metering_2, min value: 0.0, max value: 80.0
dim: Sub_metering_3, min value: 0.0, max value: 31.0
{'timestamp': (500432.15409359656, 101559890.41817391), 'Global_reactive_power': (0.1808090845853108, 1.0719119091600007), 'Voltage': (228.09332298771443, 250.039093750266), 'Global_intensity': (7.299955671108347, 44.22757691546139), 'Sub_metering_1': (4.100813784807035, 70.4555929549672), 'Sub_metering_2': (2.000261081047654, 75.51485320683649), 'Sub_metering_3': (1.2338795169456358, 29.045487646186864)}


Build query log (around 6 and a half mins), the main task is SUM.

In [5]:
from query_generate import generate_power_query_log

task = "SUM"
# number of queries
num_queries = 2000

avg_exact, query_log = generate_power_query_log(agg_col, num_queries, data, sample, dimensions, full_data_size, avg_exact=0)

query_log = generate_power_query_log(agg_col, num_queries, data, sample, dimensions, full_data_size, avg_exact=avg_exact)

Generating power query log...
Average exact sum: avg_exact=10179.321881000002
Generated 2000 queries

Generating power query log...
Generated 2000 queries



In [6]:
# Check any query in query log.
query_log[100]

{'query': {'timestamp': (567394.6235921425, 112040608.97139555),
  'Global_reactive_power': (0.12885624734795215, 1.3083444860901698),
  'Voltage': (223.4903465049087, 248.5857109877128),
  'Global_intensity': (4.186484462065307, 44.52119515412777),
  'Sub_metering_1': (21.664006451278212, 79.76330282589862),
  'Sub_metering_2': (17.0608554355749, 74.25787870809044),
  'Sub_metering_3': (2.177161742481638, 29.24967614929311)},
 'exact': 11843.328000000001,
 'estimate': 10896.021760000001,
 'error': 947.3062399999999}

New Query

In [7]:
new_query = generate_random_query(data, dimensions, test = True)
print(new_query)

dim: timestamp, min value: 0.0, max value: 124515480.0
dim: Global_reactive_power, min value: 0.0, max value: 1.39
dim: Voltage, min value: 223.2, max value: 254.15
dim: Global_intensity, min value: 0.2, max value: 48.4
dim: Sub_metering_1, min value: 0.0, max value: 88.0
dim: Sub_metering_2, min value: 0.0, max value: 80.0
dim: Sub_metering_3, min value: 0.0, max value: 31.0
{'timestamp': (6821247.849463048, 105460691.73310533), 'Global_reactive_power': (0.12981257634533902, 1.0668367680569997), 'Voltage': (224.7201371172049, 248.77815608989175), 'Global_intensity': (6.07143484432308, 40.293564718288685), 'Sub_metering_1': (2.7097010297235986, 82.78681864008222), 'Sub_metering_2': (11.199969729837015, 66.27201818962135), 'Sub_metering_3': (3.8327578693923092, 27.678752417376984)}


AQP++: no training, no prediction

q_old is selected by range similarity

In [8]:
def range_distance(q1, q2, dimensions):
    dist = 0.0
    for dim in dimensions:
        l1, r1 = q1[dim]
        l2, r2 = q2[dim]
        dist += abs(l1 - l2) + abs(r1 - r2)
    return dist

min_dist = float('inf')
opt_entry = None

for entry in query_log:
    dist = range_distance(new_query, entry['query'], dimensions)
    if dist < min_dist:
        min_dist = dist
        opt_entry = entry


In [ ]:
from query_calculate import sample_sum, exact_sum

q_old_exact  = opt_entry['exact']                # exact_sum(...)
q_hat_new    = sample_sum(agg_col, new_query, sample, full_data_size)
q_hat_old    = sample_sum(agg_col, opt_entry['query'], sample, full_data_size)

aqp_pp_estimate = q_old_exact + (q_hat_new - q_hat_old)
print("AQP++ estimate:", aqp_pp_estimate)

exact = exact_sum(agg_col, new_query, data)
print(f"Exact sum: {exact:.2f}")
print(f"Relative error: {abs(aqp_pp_estimate - exact) / exact:.4f}")


AQP++ estimate: 19007.64224
Exact sum: 10965.44
Relative error: 0.7334


In [ ]:
evaluation_query_log = generate_power_query_log(agg_col, 100, data, sample, dimensions, full_data_size, avg_exact=avg_exact)

In [ ]:
import matplotlib.pyplot as plt

# 設定數據
experiments = ['SAQP', 'AQP++', 'LAQP']
errors = [ ,0.7334, 0.03]

# 將數據按數值由大到小排序，使圖表更易讀
data = sorted(zip(experiments, errors), key=lambda x: x[1], reverse=True)
sorted_experiments, sorted_errors = zip(*data)

# 繪製條形圖
# 根據規範，不使用 plt.figure()
plt.bar(sorted_experiments, sorted_errors, color=['#3498db', '#e74c3c'])

# 設定標籤與標題，並使用 LaTeX 格式
plt.xlabel('Experiment')
plt.ylabel(r'$Relative \ Error$')
plt.title(r'$Comparison \ of \ Relative \ Error \ (AQP++ \ vs \ LAQP)$')

# 加上數值標註（選填，方便直接看結果）
for i, v in enumerate(sorted_errors):
    plt.text(i, v + 0.01, str(v), ha='center', fontweight='bold')

# 儲存圖片
plt.savefig('relative_error_comparison.png')